# S3-03: 다중 도구, 배치 처리, 구조화된 출력
**Skilljar L09-L11: Using Multiple Tools / The Batch Tool / Tools for Structured Data**

## 학습 목표
- 여러 도구를 동시에 등록하고 Claude가 자율적으로 선택하게 한다
- 배치 패턴으로 여러 항목을 일괄 처리한다
- Tool Use를 활용하여 구조화된 JSON 출력을 강제한다
- `tool_choice` 파라미터로 도구 호출을 제어한다

## 사전 준비
`.env` 파일에 API 키가 설정되어 있어야 합니다.

In [ ]:
%pip install anthropic python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json
import math

client = Anthropic()
model = "claude-sonnet-4-0"
print("설정 완료")

In [ ]:
# Tool Use 루프 함수 (S3_02에서 재사용)
def run_tool_loop(user_message, tools, tool_map, system=None, tool_choice=None):
    """Tool Use 루프를 실행하고 최종 텍스트 응답을 반환한다."""
    messages = [{"role": "user", "content": user_message}]
    
    for i in range(10):
        params = {"model": model, "max_tokens": 4096, "tools": tools, "messages": messages}
        if system:
            params["system"] = system
        if tool_choice:
            params["tool_choice"] = tool_choice
        
        response = client.messages.create(**params)
        
        if response.stop_reason == "end_turn":
            return "".join(b.text for b in response.content if b.type == "text")
        elif response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    func = tool_map.get(block.name)
                    try:
                        result = func(**block.input) if func else {"error": f"Unknown: {block.name}"}
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result, ensure_ascii=False)
                        })
                        print(f"  [{block.name}] -> {result}")
                    except Exception as e:
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(e), "is_error": True
                        })
            messages.append({"role": "user", "content": tool_results})
        else:
            break
    return "최대 반복 초과"

## 1. 다중 도구 등록

여러 도구를 `tools` 리스트에 동시에 등록하면, Claude가 질문에 따라 적절한 도구를 **자율적으로 선택**한다.

In [ ]:
# 도구 1: 덧셈
def add(a: float, b: float) -> dict:
    return {"result": a + b, "operation": "add"}

# 도구 2: 곱셈
def multiply(a: float, b: float) -> dict:
    return {"result": a * b, "operation": "multiply"}

# 도구 3: 제곱근
def sqrt(value: float) -> dict:
    return {"result": round(math.sqrt(value), 6), "operation": "sqrt"}

math_tools = [
    {"name": "add", "description": "두 수를 더한다.", "input_schema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]}},
    {"name": "multiply", "description": "두 수를 곱한다.", "input_schema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]}},
    {"name": "sqrt", "description": "제곱근을 구한다.", "input_schema": {"type": "object", "properties": {"value": {"type": "number"}}, "required": ["value"]}}
]
math_map = {"add": add, "multiply": multiply, "sqrt": sqrt}

# Claude가 질문에 따라 적절한 도구를 선택
print("=== 덧셈 질문 ===")
r1 = run_tool_loop("15 더하기 27은?", math_tools, math_map)
print(f"응답: {r1}")

print("\n=== 제곱근 질문 ===")
r2 = run_tool_loop("144의 제곱근은?", math_tools, math_map)
print(f"응답: {r2}")

## 2. tool_choice로 도구 호출 제어

| tool_choice | 설명 |
|---|---|
| `{"type": "auto"}` | Claude가 판단 (기본값) |
| `{"type": "any"}` | 반드시 도구 호출 |
| `{"type": "tool", "name": "..."}` | 특정 도구 강제 |
| `{"type": "none"}` | 도구 사용 금지 |

In [ ]:
# tool_choice 비교 실험

# auto: Claude가 도구 불필요하다고 판단하면 텍스트로 응답
resp_auto = client.messages.create(
    model=model, max_tokens=512, tools=math_tools,
    tool_choice={"type": "auto"},
    messages=[{"role": "user", "content": "안녕하세요"}]
)
print(f"auto + '안녕' → stop_reason: {resp_auto.stop_reason}")

# any: 반드시 도구를 호출
resp_any = client.messages.create(
    model=model, max_tokens=512, tools=math_tools,
    tool_choice={"type": "any"},
    messages=[{"role": "user", "content": "3과 5에 대해 뭔가 해줘"}]
)
print(f"any → stop_reason: {resp_any.stop_reason}")
for b in resp_any.content:
    if b.type == "tool_use":
        print(f"  선택된 도구: {b.name}")

# tool: 특정 도구 강제
resp_tool = client.messages.create(
    model=model, max_tokens=512, tools=math_tools,
    tool_choice={"type": "tool", "name": "multiply"},
    messages=[{"role": "user", "content": "3과 5에 대해 뭔가 해줘"}]
)
print(f"tool(multiply) → stop_reason: {resp_tool.stop_reason}")
for b in resp_tool.content:
    if b.type == "tool_use":
        print(f"  강제 호출: {b.name}({b.input})")

## 3. Tool Use로 구조화된 JSON 출력 강제

실제로 실행하지 않는 "출력 전용" 도구를 정의하고, `tool_choice`로 강제 호출하면 **스키마에 맞는 구조화된 JSON**을 얻을 수 있다.

In [ ]:
# 구조화된 출력 도구 (실행하지 않음, 입력 자체가 출력)
output_tool = {
    "name": "output_result",
    "description": "분석 결과를 구조화된 형식으로 출력한다. 분석이 완료되면 반드시 이 도구를 호출하라.",
    "input_schema": {
        "type": "object",
        "properties": {
            "summary": {"type": "string", "description": "분석 요약"},
            "score": {"type": "number", "description": "점수 (0-100)"},
            "category": {"type": "string", "enum": ["good", "average", "poor"]},
            "details": {
                "type": "array",
                "items": {"type": "string"},
                "description": "세부 항목 리스트"
            }
        },
        "required": ["summary", "score", "category", "details"]
    }
}

# tool_choice로 이 도구 강제 호출 → input이 구조화된 출력
resp = client.messages.create(
    model=model, max_tokens=1024,
    tools=[output_tool],
    tool_choice={"type": "tool", "name": "output_result"},
    messages=[{"role": "user", "content": "Python 프로그래밍 언어의 장단점을 분석해줘."}]
)

for block in resp.content:
    if block.type == "tool_use":
        structured_output = block.input
        print(json.dumps(structured_output, indent=2, ensure_ascii=False))

---
## 연습 1: 다중 도구 챗봇

날씨 + 계산기 + 단위변환 3개 도구를 등록하고, Claude가 질문에 따라 적절한 도구를 선택하게 하세요.

**요구사항:**
1. 3개 도구 함수와 스키마 정의
2. `run_tool_loop`으로 3가지 다른 질문 처리
3. 각 질문에서 올바른 도구가 선택되었는지 확인

In [ ]:
# TODO: 3개 도구 등록 + 다중 질문 처리

In [ ]:
# ===== 정답 =====

def get_weather(city):
    data = {"서울": {"temp": 15, "condition": "맑음"}, "부산": {"temp": 18, "condition": "구름"}}
    return data.get(city, {"temp": 0, "condition": "알 수 없음"})

def calculator(a, b, op):
    ops = {"add": a+b, "subtract": a-b, "multiply": a*b, "divide": a/b if b else None}
    return {"result": ops.get(op, None)}

def convert_unit(value, from_unit, to_unit):
    conv = {("mm","m"): value/1000, ("m","mm"): value*1000, ("kN","N"): value*1000, ("N","kN"): value/1000}
    return {"result": conv.get((from_unit, to_unit), None), "from": from_unit, "to": to_unit}

multi_tools = [
    {"name": "get_weather", "description": "도시의 날씨를 조회한다.", "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}},
    {"name": "calculator", "description": "사칙연산을 수행한다.", "input_schema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}, "op": {"type": "string", "enum": ["add","subtract","multiply","divide"]}}, "required": ["a","b","op"]}},
    {"name": "convert_unit", "description": "건축공학 단위를 변환한다 (mm<->m, kN<->N).", "input_schema": {"type": "object", "properties": {"value": {"type": "number"}, "from_unit": {"type": "string"}, "to_unit": {"type": "string"}}, "required": ["value","from_unit","to_unit"]}}
]
multi_map = {"get_weather": get_weather, "calculator": calculator, "convert_unit": convert_unit}

print("=== 질문 1: 날씨 ===")
r1 = run_tool_loop("서울 날씨 어때?", multi_tools, multi_map)
print(f"답: {r1[:100]}\n")

print("=== 질문 2: 계산 ===")
r2 = run_tool_loop("125 곱하기 48은?", multi_tools, multi_map)
print(f"답: {r2[:100]}\n")

print("=== 질문 3: 단위변환 ===")
r3 = run_tool_loop("500mm를 m로 변환해줘", multi_tools, multi_map)
print(f"답: {r3[:100]}")

def verify():
    assert all(isinstance(r, str) and len(r) > 0 for r in [r1, r2, r3])
    print("모든 검증 통과! 3개 도구 모두 적절히 선택됨!")

verify()

---
## 연습 2: tool_choice 실험

동일한 질문에 대해 `tool_choice`를 변경하면서 Claude의 행동 변화를 관찰하세요.

**요구사항:**
1. `auto`: Claude가 도구 사용 여부를 판단
2. `any`: 반드시 도구 호출
3. `tool` (특정 도구 지정): 특정 도구 강제
4. 각 경우의 `stop_reason`과 응답 내용 비교

In [ ]:
# TODO: tool_choice 실험

In [ ]:
# ===== 정답 =====

question = "3과 7에 대해 알려줘"
results = {}

# auto
resp_a = client.messages.create(model=model, max_tokens=512, tools=math_tools, tool_choice={"type": "auto"}, messages=[{"role": "user", "content": question}])
results["auto"] = resp_a.stop_reason
print(f"auto → {resp_a.stop_reason}")

# any
resp_b = client.messages.create(model=model, max_tokens=512, tools=math_tools, tool_choice={"type": "any"}, messages=[{"role": "user", "content": question}])
results["any"] = resp_b.stop_reason
tool_name = next((b.name for b in resp_b.content if b.type == "tool_use"), None)
print(f"any → {resp_b.stop_reason} (도구: {tool_name})")

# tool (multiply 강제)
resp_c = client.messages.create(model=model, max_tokens=512, tools=math_tools, tool_choice={"type": "tool", "name": "multiply"}, messages=[{"role": "user", "content": question}])
results["tool"] = resp_c.stop_reason
forced_name = next((b.name for b in resp_c.content if b.type == "tool_use"), None)
print(f"tool(multiply) → {resp_c.stop_reason} (도구: {forced_name})")

def verify():
    assert results["any"] == "tool_use", "any는 반드시 tool_use"
    assert results["tool"] == "tool_use", "tool은 반드시 tool_use"
    assert forced_name == "multiply", "tool 모드에서는 지정한 도구만 호출"
    print("모든 검증 통과! tool_choice별 행동 차이 확인!")

verify()

---
## 건축공학 실습: 구조 부재 검토 보고서 JSON 출력

### 과제: Tool Use로 구조화된 보 검토 보고서를 JSON으로 출력하세요.

**요구사항:**
1. 보 검토 보고서 스키마를 정의 (beam_id, section, flexure, shear, overall)
2. `tool_choice={"type": "tool", "name": "output_beam_report"}`로 강제 호출
3. Claude가 입력 데이터를 분석하여 스키마에 맞는 JSON을 생성
4. 생성된 JSON을 파싱하여 판정 결과 확인

**검토 조건:**
- B1: 350x600, fck=27MPa, fy=400MPa
- 주근 5-D25(2540mm2), 전단보강근 D10@200(Av=142.6mm2)
- Mu=380kN-m, Vu=250kN

In [ ]:
# TODO: 구조화된 출력 도구로 보 검토 보고서 생성

In [ ]:
# ===== 정답 =====

beam_report_tool = {
    "name": "output_beam_report",
    "description": "RC 보의 설계 검토 결과를 구조화된 형식으로 출력한다. 검토가 완료되면 이 도구를 호출하라.",
    "input_schema": {
        "type": "object",
        "properties": {
            "beam_id": {"type": "string", "description": "보 ID"},
            "section": {
                "type": "object",
                "properties": {
                    "b_mm": {"type": "number"},
                    "d_mm": {"type": "number"}
                }
            },
            "flexure": {
                "type": "object",
                "properties": {
                    "Mu_kNm": {"type": "number", "description": "설계 모멘트"},
                    "phi_Mn_kNm": {"type": "number", "description": "설계 모멘트 강도"},
                    "result": {"type": "string", "enum": ["OK", "NG"]}
                }
            },
            "shear": {
                "type": "object",
                "properties": {
                    "Vu_kN": {"type": "number", "description": "설계 전단력"},
                    "phi_Vn_kN": {"type": "number", "description": "설계 전단 강도"},
                    "result": {"type": "string", "enum": ["OK", "NG"]}
                }
            },
            "overall": {"type": "string", "enum": ["PASS", "FAIL"]}
        },
        "required": ["beam_id", "section", "flexure", "shear", "overall"]
    }
}

resp = client.messages.create(
    model=model, max_tokens=2048,
    tools=[beam_report_tool],
    tool_choice={"type": "tool", "name": "output_beam_report"},
    messages=[{
        "role": "user",
        "content": (
            "다음 RC 보를 검토하고 결과를 출력하라.\n"
            "B1: 350x600, fck=27MPa, fy=400MPa\n"
            "주근 5-D25(As=2540mm2), 전단보강근 D10@200(Av=142.6mm2)\n"
            "Mu=380kN-m, Vu=250kN\n"
            "KDS 14 20 기준으로 휨과 전단을 계산하여 판정하라."
        )
    }]
)

# 구조화된 출력 추출
report = None
for block in resp.content:
    if block.type == "tool_use":
        report = block.input
        break

print("=== 보 검토 보고서 (JSON) ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

def verify():
    assert report is not None, "보고서가 생성되어야 함"
    assert report["beam_id"] == "B1", f"beam_id 확인: {report['beam_id']}"
    assert report["overall"] in ("PASS", "FAIL"), f"overall 확인: {report['overall']}"
    assert "flexure" in report, "휨 검토 포함"
    assert "shear" in report, "전단 검토 포함"
    print(f"모든 검증 통과! 종합 판정: {report['overall']}")

verify()